# Tracing with LangSmith and LangGraph Studio

**LangSmith** is LangChain's observability platform for tracing, evaluating, and monitoring LLM applications. **LangGraph Studio** is a visual IDE for designing, running, and debugging LangGraph agents — it reads your compiled graph and displays node-by-node execution with LangSmith traces attached.

```
  LangGraph Studio UI          LangSmith Project
  ───────────────────          ─────────────────
  [load_memory] ──►            Run: shopco-support
  [route]       ──►              ├─ chain: load_memory
  [respond]     ──►              ├─ chain: route_intent
  [persist]     ──►              ├─ llm: compose_answer
                                 └─ chain: persist_memory
```

| Tool | Role |
|------|------|
| **LangSmith** | Trace storage, eval datasets, feedback scores |
| **LangGraph** | Agent graph with automatic run nesting |
| **LangGraph Studio** | Visual graph editor + step-through debugger |
| **`@traceable`** | Manual tracing for custom Python functions |

This notebook builds **ShopCo Support** as a traced LangGraph workflow with:

1. Offline `ShopCoLangSmith` run tree (no API key required)
2. `@traceable` decorator and `tracing_context`
3. LangGraph `StateGraph` with per-node runs
4. RunnableConfig metadata (`user_id`, `session_id`)
5. Studio graph manifest and debugging patterns

Offline by default; optional live LangSmith at the end.

In [ ]:
"""
ShopCo Support — LangSmith + LangGraph Studio tracing lab.
"""

import json
import os
import re
import time
import uuid
from contextlib import contextmanager
from contextvars import ContextVar
from dataclasses import dataclass, field
from datetime import datetime, timezone
from functools import wraps
from typing import Annotated, Any, Callable, Generator, Literal

from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.runnables import RunnableConfig
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from typing_extensions import TypedDict

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "sk-proj-placeholder-replace-with-your-real-key")
os.environ.setdefault("OPENAI_API_KEY", OPENAI_API_KEY)

USE_LIVE_LLM = False
USE_LIVE_LANGSMITH = False

LANGSMITH_API_KEY = os.environ.get("LANGSMITH_API_KEY", "ls-placeholder-key")
LANGSMITH_PROJECT = os.environ.get("LANGSMITH_PROJECT", "shopco-support")
os.environ.setdefault("LANGSMITH_TRACING", "false")


def pretty(obj: Any) -> str:
    return json.dumps(obj, indent=2, default=str)


def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()


POLICY_SNIPPETS = {
    "returns": "Returns within 30 days. Refunds in 5-7 business days. [pol-returns-01]",
    "shipping": "Free shipping over $50. [pol-shipping-02]",
}

ORDERS_DB = {
    "ORD-1001": {"order_id": "ORD-1001", "status": "shipped"},
    "ORD-1002": {"order_id": "ORD-1002", "status": "processing"},
}

print("ShopCo LangSmith lab ready.")

---

## 1. LangSmith Run Model

LangSmith stores **runs** in a tree (similar to OTEL spans):

| `run_type` | Use for |
|------------|--------|
| `chain` | Graph nodes, orchestration steps |
| `llm` | Model calls |
| `tool` | Tool invocations |
| `retriever` | RAG retrieval |
| `embedding` | Embedding calls |

Each run has `inputs`, `outputs`, `error`, `latency`, `tokens`, and optional `metadata`.

---

## 2. Offline ShopCoLangSmith — Run Tree

In [ ]:
RunType = Literal["chain", "llm", "tool", "retriever", "embedding"]

_current_run: ContextVar[str | None] = ContextVar("current_run", default=None)
_tracing_project: ContextVar[str] = ContextVar("tracing_project", default="shopco-support")
_tracing_enabled: ContextVar[bool] = ContextVar("tracing_enabled", default=True)
_run_metadata: ContextVar[dict[str, Any]] = ContextVar("run_metadata", default={})


@dataclass
class RunRecord:
    id: str
    name: str
    run_type: RunType
    parent_id: str | None
    project: str
    inputs: Any = None
    outputs: Any = None
    error: str = ""
    start_time: str = ""
    end_time: str = ""
    latency_ms: float = 0.0
    metadata: dict[str, Any] = field(default_factory=dict)
    token_usage: dict[str, int] = field(default_factory=dict)
    feedback: list[dict[str, Any]] = field(default_factory=list)


class ShopCoLangSmith:
    """In-memory LangSmith-compatible run store for offline demos."""

    def __init__(self) -> None:
        self.runs: dict[str, RunRecord] = {}
        self._root_id: str | None = None

    @contextmanager
    def start_run(
        self,
        *,
        name: str,
        run_type: RunType = "chain",
        inputs: Any = None,
        metadata: dict[str, Any] | None = None,
    ) -> Generator[RunRecord, None, None]:
        if not _tracing_enabled.get():
            yield RunRecord(id="noop", name=name, run_type=run_type, parent_id=None, project="")
            return

        parent = _current_run.get()
        rid = f"run-{uuid.uuid4().hex[:12]}"
        if not parent:
            self._root_id = rid
        merged_meta = {**_run_metadata.get(), **(metadata or {})}
        run = RunRecord(
            id=rid, name=name, run_type=run_type, parent_id=parent,
            project=_tracing_project.get(), inputs=inputs,
            start_time=utc_now(), metadata=merged_meta,
        )
        self.runs[rid] = run
        token = _current_run.set(rid)
        t0 = time.perf_counter()
        try:
            yield run
        except Exception as exc:
            run.error = str(exc)
            raise
        finally:
            run.end_time = utc_now()
            run.latency_ms = (time.perf_counter() - t0) * 1000
            _current_run.reset(token)

    def create_feedback(self, run_id: str, *, key: str, score: float, comment: str = "") -> None:
        if run_id in self.runs:
            self.runs[run_id].feedback.append({"key": key, "score": score, "comment": comment})

    def reset(self) -> None:
        self.runs.clear()
        self._root_id = None

    def print_run_tree(self, root_id: str | None = None) -> None:
        rid = root_id or self._root_id
        if not rid or rid not in self.runs:
            print("(no runs)")
            return
        root = self.runs[rid]
        print(f"Project: {root.project} | Root: {root.name} ({root.id})")

        def render(parent_id: str | None, indent: int = 0) -> None:
            children = sorted(
                [r for r in self.runs.values() if r.parent_id == parent_id],
                key=lambda r: r.start_time,
            )
            for r in children:
                err = " ✗" if r.error else ""
                tok = f" tokens={r.token_usage}" if r.token_usage else ""
                print(f"{'  '*indent}[{r.run_type}] {r.name} ({r.latency_ms:.1f}ms){tok}{err}")
                render(r.id, indent + 1)

        render(rid)


ls_client = ShopCoLangSmith()
print("ShopCoLangSmith ready.")

---

## 3. `@traceable` and `tracing_context`

In [ ]:
@contextmanager
def tracing_context(
    *,
    project_name: str = "shopco-support",
    enabled: bool = True,
    metadata: dict[str, Any] | None = None,
) -> Generator[None, None, None]:
    """Offline mirror of langsmith.tracing_context."""
    tokens = [
        _tracing_project.set(project_name),
        _tracing_enabled.set(enabled),
        _run_metadata.set(metadata or {}),
    ]
    try:
        yield
    finally:
        _tracing_project.reset(tokens[0])
        _tracing_enabled.reset(tokens[1])
        _run_metadata.reset(tokens[2])


def traceable(
    func: Callable | None = None,
    *,
    name: str | None = None,
    run_type: RunType = "chain",
) -> Callable:
    """Offline mirror of langsmith.traceable decorator."""

    def decorator(fn: Callable) -> Callable:
        run_name = name or fn.__name__

        @wraps(fn)
        def wrapper(*args: Any, **kwargs: Any) -> Any:
            with ls_client.start_run(name=run_name, run_type=run_type, inputs={"args": args, "kwargs": kwargs}) as run:
                result = fn(*args, **kwargs)
                run.outputs = result
                return result

        return wrapper

    if func is not None:
        return decorator(func)
    return decorator


ls_client.reset()


@traceable(name="retrieve_policy", run_type="retriever")
def retrieve_policy(query: str) -> dict[str, str]:
    if "return" in query.lower():
        return {"id": "pol-returns-01", "text": POLICY_SNIPPETS["returns"]}
    return {"id": "none", "text": "No policy found."}


@traceable(name="lookup_order", run_type="tool")
def lookup_order(query: str) -> dict[str, str]:
    m = re.search(r"ORD-[0-9]{4}", query.upper())
    if not m:
        return {"order_id": "", "status": "unknown"}
    return ORDERS_DB.get(m.group(0), {"order_id": m.group(0), "status": "not_found"})


@traceable(name="compose_answer", run_type="llm")
def compose_answer(question: str, context: dict[str, Any]) -> str:
    rid = _current_run.get()
    if rid and rid in ls_client.runs:
        ls_client.runs[rid].token_usage = {"prompt_tokens": 100, "completion_tokens": 40, "total_tokens": 140}
    if context.get("policy"):
        return context["policy"]["text"]
    if context.get("order"):
        o = context["order"]
        return f"Order {o['order_id']} status: {o['status']}."
    return "How can I help?"


@traceable(name="handle_support_request")
def handle_support_request(question: str) -> str:
    ctx: dict[str, Any] = {}
    if "return" in question.lower():
        ctx["policy"] = retrieve_policy(question)
    elif "order" in question.lower() or re.search(r"ORD-[0-9]{4}", question.upper()):
        ctx["order"] = lookup_order(question)
    return compose_answer(question, ctx)


with tracing_context(project_name="shopco-demo", metadata={"user_id": "alice", "session_id": "s-1"}):
    print(handle_support_request("What is the return policy?"))
ls_client.print_run_tree()

---

## 4. LangGraph State — ShopCo Support

In [ ]:
class SupportState(TypedDict):
    messages: Annotated[list, add_messages]
    intent: str
    order_id: str
    trace_metadata: dict[str, Any]


def traced_node(name: str, run_type: RunType = "chain"):
    """Wrap a LangGraph node so each invocation creates a LangSmith run."""

    def decorator(fn: Callable) -> Callable:
        @wraps(fn)
        def wrapper(state: SupportState, config: RunnableConfig) -> dict[str, Any]:
            meta = config.get("configurable", {})
            with ls_client.start_run(
                name=f"lg:{name}", run_type=run_type,
                inputs={"state_keys": list(state.keys()), "configurable": meta},
                metadata={"node": name, "thread_id": meta.get("thread_id", "")},
            ) as run:
                result = fn(state, config)
                run.outputs = {k: str(v)[:100] for k, v in result.items()}
                return result

        return wrapper

    return decorator

---

## 5. Traced LangGraph Nodes

In [ ]:
@traced_node("route_intent")
def route_node(state: SupportState, config: RunnableConfig) -> dict[str, Any]:
    last = next((m for m in reversed(state["messages"]) if isinstance(m, HumanMessage)), None)
    q = str(last.content).lower() if last else ""
    if state.get("order_id") and ("status" in q or " it" in q or q.startswith("it")):
        return {"intent": "order"}
    if "return" in q or "refund" in q:
        return {"intent": "policy"}
    if "order" in q or re.search(r"ORD-[0-9]{4}", str(last.content if last else "").upper()):
        return {"intent": "order"}
    return {"intent": "general"}


@traced_node("retrieve", run_type="retriever")
def retrieve_node(state: SupportState, config: RunnableConfig) -> dict[str, Any]:
    last = next((m for m in reversed(state["messages"]) if isinstance(m, HumanMessage)), None)
    q = str(last.content) if last else ""
    ctx: dict[str, Any] = {}
    if state.get("intent") == "policy":
        ctx["policy"] = retrieve_policy(q)
    elif state.get("intent") == "order":
        oid = state.get("order_id") or ""
        if not re.search(r"ORD-[0-9]{4}", q.upper()) and oid:
            q_lookup = oid
        else:
            q_lookup = q
        row = lookup_order(q_lookup)
        ctx["order"] = row
        return {"order_id": row.get("order_id", oid), "trace_metadata": ctx}
    return {"trace_metadata": ctx}


@traced_node("respond", run_type="llm")
def respond_node(state: SupportState, config: RunnableConfig) -> dict[str, Any]:
    last = next((m for m in reversed(state["messages"]) if isinstance(m, HumanMessage)), None)
    q = str(last.content) if last else ""
    ctx = state.get("trace_metadata", {})
    answer = compose_answer(q, ctx)
    rid = _current_run.get()
    if rid and rid in ls_client.runs:
        ls_client.runs[rid].token_usage = {"total_tokens": 140}
    return {"messages": [AIMessage(content=answer)]}

---

## 6. Compile Graph — Studio-Ready

In [ ]:
def build_support_graph():
    builder = StateGraph(SupportState)
    builder.add_node("route_intent", route_node)
    builder.add_node("retrieve", retrieve_node)
    builder.add_node("respond", respond_node)

    builder.add_edge(START, "route_intent")
    builder.add_edge("route_intent", "retrieve")
    builder.add_edge("retrieve", "respond")
    builder.add_edge("respond", END)

    return builder.compile(checkpointer=MemorySaver())


GRAPH = build_support_graph()


@dataclass
class StudioGraphManifest:
    """What LangGraph Studio displays for a compiled graph."""
    graph_id: str
    nodes: list[str]
    edges: list[tuple[str, str]]
    state_schema: list[str]
    checkpointer: str


STUDIO_MANIFEST = StudioGraphManifest(
    graph_id="shopco-support",
    nodes=["route_intent", "retrieve", "respond"],
    edges=[("__start__", "route_intent"), ("route_intent", "retrieve"), ("retrieve", "respond"), ("respond", "__end__")],
    state_schema=["messages", "intent", "order_id", "trace_metadata"],
    checkpointer="MemorySaver",
)

print("LangGraph compiled. Studio manifest:")
print(pretty(STUDIO_MANIFEST))

---

## 7. Invoke with RunnableConfig — Thread + Metadata

In [ ]:
def invoke_support(
    graph,
    question: str,
    *,
    user_id: str,
    thread_id: str,
    project: str = "shopco-support",
) -> SupportState:
    ls_client.reset()
    config: RunnableConfig = {
        "configurable": {
            "thread_id": thread_id,
            "user_id": user_id,
        },
        "metadata": {
            "langsmith_project": project,
            "user_id": user_id,
            "session_id": thread_id,
        },
        "run_name": "shopco-support-turn",
    }
    with tracing_context(project_name=project, metadata={"user_id": user_id, "thread_id": thread_id}):
        with ls_client.start_run(name="graph_invoke", run_type="chain", inputs={"question": question}):
            return graph.invoke({"messages": [HumanMessage(content=question)]}, config=config)


state = invoke_support(GRAPH, "What is the return policy?", user_id="alice", thread_id="chat-42")
print(state["messages"][-1].content)
print()
ls_client.print_run_tree()

---

## 8. LangGraph Studio — What You See

LangGraph Studio connects to your **compiled graph** and provides:

| Studio feature | Purpose |
|----------------|--------|
| **Graph canvas** | Visual nodes and edges from manifest |
| **State inspector** | Live `SupportState` after each step |
| **Step-through** | Run one node at a time |
| **Time travel** | Replay from checkpointer history |
| **LangSmith link** | Open trace for current run |

Studio expects a `langgraph.json` config pointing at your graph factory. The manifest above is what Studio renders for `shopco-support`.

In [ ]:
LANGGRAPH_JSON_EXAMPLE = {
    "graphs": {
        "shopco_support": {
            "path": "./graph_module.py:build_support_graph",
            "description": "ShopCo tier-1 support agent",
        }
    },
    "env": ".env",
    "dependencies": ["./"],
}

print("langgraph.json structure (for Studio):")
print(pretty(LANGGRAPH_JSON_EXAMPLE))

---

## 9. Multi-Turn Thread with Checkpoint + Trace

In [ ]:
cfg = {"configurable": {"thread_id": "chat-99", "user_id": "alice"}}

with tracing_context(project_name="shopco-support", metadata={"user_id": "alice"}):
    ls_client.reset()
    with ls_client.start_run(name="turn_1", inputs={"q": "My order is ORD-1001"}):
        s1 = GRAPH.invoke({"messages": [HumanMessage(content="My order is ORD-1001")]}, config=cfg)
    with ls_client.start_run(name="turn_2", inputs={"q": "What's its status?"}):
        s2 = GRAPH.invoke({"messages": [HumanMessage(content="What's its status?")]}, config=cfg)

print("Turn 2 answer:", s2["messages"][-1].content)
print(f"Checkpoint preserved order_id: {s2.get('order_id', '(from retrieve)')}")
print(f"Total runs logged: {len(ls_client.runs)}")

---

## 10. Error Runs and Debugging

In [ ]:
@traceable(name="failing_lookup", run_type="tool")
def failing_lookup(order_id: str) -> str:
    if order_id == "ORD-9999":
        raise ValueError("Warehouse API timeout")
    return "ok"


ls_client.reset()
try:
    with tracing_context(project_name="shopco-errors"):
        with ls_client.start_run(name="error_demo", inputs={"order_id": "ORD-9999"}):
            failing_lookup("ORD-9999")
except ValueError:
    pass

error_runs = [r for r in ls_client.runs.values() if r.error]
print("Failed runs:", [(r.name, r.error) for r in error_runs])
ls_client.print_run_tree()

---

## 11. Feedback Scores on Runs

In [ ]:
state = invoke_support(GRAPH, "ORD-1001 status", user_id="bob", thread_id="fb-1")
root_id = ls_client._root_id
if root_id:
    ls_client.create_feedback(root_id, key="task_complete", score=1.0, comment="correct status")
    ls_client.create_feedback(root_id, key="user_rating", score=0.9, comment="helpful")

if root_id and root_id in ls_client.runs:
    print("Feedback:", ls_client.runs[root_id].feedback)

---

## 12. ReleaseFlow — Traced Deploy Graph

In [ ]:
class DeployState(TypedDict):
    service: str
    version: str
    tests_passed: bool
    canary_ok: bool
    message: str


@traced_node("run_tests")
def tests_node(state: DeployState, config: RunnableConfig) -> dict[str, Any]:
    return {"tests_passed": True, "message": "42 tests passed"}


@traced_node("deploy_canary")
def canary_node(state: DeployState, config: RunnableConfig) -> dict[str, Any]:
    return {"canary_ok": True, "message": f"{state['service']}@{state['version']} at 10%"}


@traced_node("finalize")
def finalize_node(state: DeployState, config: RunnableConfig) -> dict[str, Any]:
    if not state.get("tests_passed"):
        return {"message": "BLOCKED: tests failed"}
    return {"message": f"Deployed {state['service']}@{state['version']}"}


def build_deploy_graph():
    b = StateGraph(DeployState)
    for name, fn in [("run_tests", tests_node), ("deploy_canary", canary_node), ("finalize", finalize_node)]:
        b.add_node(name, fn)
    b.add_edge(START, "run_tests")
    b.add_edge("run_tests", "deploy_canary")
    b.add_edge("deploy_canary", "finalize")
    b.add_edge("finalize", END)
    return b.compile()


ls_client.reset()
with tracing_context(project_name="releaseflow", metadata={"team": "platform"}):
    with ls_client.start_run(name="releaseflow_deploy", inputs={"service": "shopco-api", "version": "v2.4.1"}):
        dep = build_deploy_graph()
        out = dep.invoke({"service": "shopco-api", "version": "v2.4.1", "tests_passed": False, "canary_ok": False, "message": ""})
print(out["message"])
ls_client.print_run_tree()

---

## 13. LangSmith vs Langfuse

| Dimension | LangSmith | Langfuse |
|-----------|-----------|----------|
| Ecosystem | Native LangChain/LangGraph | Framework-agnostic |
| Studio | LangGraph Studio integration | Separate UI |
| Auto-trace LangGraph | Yes (env vars) | Via CallbackHandler |
| Eval datasets | Built-in | Built-in |
| Self-host | LangSmith deployment options | Open source |

Use **LangSmith** when your stack is LangGraph-first. Use **Langfuse** for polyglot or self-hosted OSS preference.

---

## 14. Trace Nesting Troubleshooting

| Issue | Cause | Fix |
|-------|-------|-----|
| Flat traces (no children) | Python < 3.11 async context | Upgrade Python or pass `config` to nodes |
| Disconnected tool runs | Custom fn not `@traceable` | Decorate tools and retrievers |
| Missing metadata | Not in RunnableConfig | Set `config["metadata"]` |
| Wrong project | Default project | `tracing_context(project_name=...)` |

---

## 15. Eval Harness

In [ ]:
ls_client.reset()
state = invoke_support(GRAPH, "Return policy?", user_id="eval", thread_id="eval-1")

EVAL = [
    ("root run created", lambda: ls_client._root_id is not None),
    ("langgraph nodes traced", lambda: sum(1 for r in ls_client.runs.values() if r.name.startswith("lg:")) >= 3),
    ("llm run has tokens", lambda: any(r.token_usage for r in ls_client.runs.values())),
    ("studio manifest nodes", lambda: len(STUDIO_MANIFEST.nodes) == 3),
    ("answer generated", lambda: "return" in state["messages"][-1].content.lower()),
]

passed = sum(int(fn()) for _, fn in EVAL)
for label, fn in EVAL:
    print(f"{'✓' if fn() else '✗'} {label}")
print(f"\nEval: {passed}/{len(EVAL)}")

---

## 16. Common Mistakes

| Mistake | Symptom | Fix |
|---------|---------|-----|
| No `LANGSMITH_TRACING` | Empty LangSmith UI | Set env vars |
| Custom tools untraced | Gaps in graph trace | `@traceable(run_type="tool")` |
| One project for all envs | Prod/test mixed | `LANGSMITH_PROJECT` per env |
| Ignoring RunnableConfig metadata | Can't filter by user | `metadata={"user_id": ...}` |
| No feedback scores | Can't rank runs | `create_feedback` on root run |

---

## 17. Production Checklist

- [ ] `LANGSMITH_TRACING=true`, `LANGSMITH_API_KEY`, `LANGSMITH_PROJECT`
- [ ] LangGraph graph in `langgraph.json` for Studio
- [ ] `@traceable` on custom tools outside LangChain
- [ ] `RunnableConfig` with `thread_id`, `user_id`, `metadata`
- [ ] Separate projects: dev / staging / prod
- [ ] Feedback scores on production runs
- [ ] Python 3.11+ for reliable async trace nesting

---

## 18. Optional Live LangSmith

In [ ]:
def demo_live_langsmith() -> None:
    if not USE_LIVE_LANGSMITH:
        print("Set USE_LIVE_LANGSMITH = True and configure LANGSMITH_API_KEY.")
        print(f"Project: {LANGSMITH_PROJECT}")
        return

    os.environ["LANGSMITH_TRACING"] = "true"
    os.environ["LANGSMITH_API_KEY"] = LANGSMITH_API_KEY
    os.environ["LANGSMITH_PROJECT"] = LANGSMITH_PROJECT

    import langsmith as ls
    from langsmith import traceable as ls_traceable

    @ls_traceable(name="live_support", run_type="chain")
    def live_support(question: str) -> str:
        return handle_support_request(question)

    with ls.tracing_context(project_name=LANGSMITH_PROJECT, enabled=True):
        result = live_support("What is the return policy?")
    print("Live trace sent:", result[:80])
    print("View in LangSmith UI under project:", LANGSMITH_PROJECT)


demo_live_langsmith()

---

## 19. Quiz

1. How does LangGraph auto-tracing work with LangSmith?
2. What does LangGraph Studio show that LangSmith alone does not?
3. When should you use `@traceable` vs relying on auto-trace?
4. What RunnableConfig fields help filter traces by user?
5. Why can Python 3.10 async graphs produce flat traces?

---

## 20. Summary

**LangSmith + LangGraph Studio** provide end-to-end agent observability for LangGraph apps:

1. **LangSmith runs** — nested tree of chain/llm/tool/retriever steps
2. **`@traceable`** — manual tracing for custom functions
3. **LangGraph** — each node becomes a traced run when wrapped
4. **Studio** — visual graph, state inspector, checkpoint time travel
5. **RunnableConfig** — thread_id, metadata, project routing

ShopCo Support and ReleaseFlow deploy graphs demonstrate offline tracing that maps 1:1 to live LangSmith when `USE_LIVE_LANGSMITH=True`.